# Week 11 — Notebook 3: K-Means Limitations

**Difficulty:** ⭐⭐⭐ | **Estimated Time:** 45 minutes

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Demonstrate why K-means fails on non-convex (ring-shaped, crescent-shaped) clusters
2. Explain the equal-variance / spherical-cluster assumption baked into K-means
3. Show how outliers drag centroids away from the true cluster center
4. Know when **not** to use K-means — and which alternatives to reach for instead

---

**Theme:** Retail customer segmentation — understanding K-means failure modes before trusting it with real business data.


## The Map Analogy

Imagine you are a city planner drawing **police patrol zones** on a map. K-means draws boundaries the same way a Voronoi diagram does — every location belongs to whichever police station is closest. Those boundaries are always **straight lines** (or flat hyperplanes in higher dimensions).

That works perfectly when the crime hotspots are compact, round blobs. But what if one hotspot is a **ring road** around the city, and another is the downtown core inside it? No matter how you place two straight-line boundaries, you cannot separate a ring from its interior — the ring always gets cut into pieces.

That is exactly K-means' first and most important limitation: **it can only separate clusters with linear (Voronoi) boundaries**. Any cluster shape that cannot be separated from its neighbours by a straight line will be mis-segmented.

The same logic applies to customer data: if "budget shoppers" and "luxury shoppers" do not form two neat, round clouds in feature space, K-means will draw the wrong line between them.


## Why This Matters for ML

K-means is the default clustering algorithm in almost every ML tutorial — it is fast, easy to explain to stakeholders, and scales to millions of rows. But it has five structural limitations that can make its output **actively misleading**:

| Limitation | Real-world consequence |
|---|---|
| Non-convex clusters | Customers who form a ring in spend-vs-frequency space get split arbitrarily |
| Equal-variance assumption | One tight 'VIP' cluster + one broad 'casual' cluster → VIP gets absorbed |
| Sensitivity to outliers | Five whale customers move the centroid; the whole segment description changes |
| Must specify k in advance | You need a separate step just to choose k |
| Local optima | Two runs of K-means on the same data can produce completely different segments |

Knowing these limitations lets you:
- **Choose the right algorithm** (DBSCAN for non-convex, GMM for unequal variance, etc.)
- **Pre-process data** to reduce limitation impact (outlier removal, feature scaling)
- **Communicate uncertainty** to stakeholders who may over-trust the colorful cluster plot


In [ ]:
# ── Imports and global settings ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import time

from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

np.random.seed(42)                        # reproducibility for all random ops
warnings.filterwarnings('ignore')         # suppress sklearn convergence warnings in demos
sns.set_theme(style='whitegrid')          # consistent plot style throughout notebook

print('All imports successful.')


---
## Limitation 1: Non-Convex Clusters

### Why K-means cannot find rings or crescents

K-means assigns each point to its **nearest centroid**. The set of all points closer to centroid A than to centroid B is separated from the set closer to B by a **straight line** — mathematically, the perpendicular bisector of the segment joining the two centroids. This boundary is called a **Voronoi cell edge**.

A ring-shaped cluster and its interior are **not linearly separable**: you cannot draw a straight line that puts all ring points on one side and all interior points on the other. Therefore K-means will always cut the ring into arcs and mix them with interior points.

The `make_moons` dataset (two interleaved crescents) and `make_circles` dataset (inner ring + outer ring) are the canonical examples of this failure.


In [ ]:
np.random.seed(42)

# ── Generate non-convex datasets ─────────────────────────────────────────────
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)    # two interleaved crescents
X_circles, y_circles = make_circles(n_samples=300, noise=0.05,
                                     factor=0.45, random_state=42)           # inner + outer ring

# ── Run KMeans (k=2) on each ──────────────────────────────────────────────────
km_moons   = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_moons)
km_circles = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_circles)

# ── Plot: true structure (left) vs KMeans result (right) ─────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

datasets  = [(X_moons, y_moons, km_moons,   'Moons'),
             (X_circles, y_circles, km_circles, 'Circles')]
palette   = ['#e41a1c', '#377eb8']          # red and blue for binary clusters

for row, (X, y_true, km, name) in enumerate(datasets):
    # Left panel — ground truth
    ax = axes[row, 0]
    for label in [0, 1]:
        mask = y_true == label
        ax.scatter(X[mask, 0], X[mask, 1], c=palette[label],
                   s=20, alpha=0.8, label=f'True cluster {label}')
    ax.set_title(f'{name} — True Structure', fontsize=12)
    ax.legend(fontsize=9)

    # Right panel — KMeans prediction
    ax = axes[row, 1]
    pred = km.labels_
    for label in [0, 1]:
        mask = pred == label
        ax.scatter(X[mask, 0], X[mask, 1], c=palette[label],
                   s=20, alpha=0.8, label=f'KMeans cluster {label}')
    # Mark centroids with an 'X'
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', marker='X', s=200, zorder=5, label='Centroids')
    ax.set_title(f'{name} — KMeans (k=2) FAILS', fontsize=12, color='crimson')
    ax.legend(fontsize=9)

fig.suptitle('Limitation 1: KMeans Cannot Separate Non-Convex Clusters',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Notice: KMeans splits each crescent/ring vertically — it cannot follow curved boundaries.')


---
## Limitation 2: Equal-Variance / Spherical Cluster Assumption

### What K-means implicitly assumes about cluster shape

K-means minimises **within-cluster sum of squares (WCSS)**:

> WCSS = Σ Σ ‖xᵢ − μₖ‖²

This objective is equivalent to fitting a **Gaussian mixture where every component has the same spherical covariance**. In plain English: K-means assumes every cluster is a round ball of approximately the same size.

Real-world clusters are rarely round or the same size:
- A small VIP segment (tight cloud of 50 high-spenders) sits next to a large mass-market segment (loose cloud of 5000 regular shoppers)
- K-means will pull the boundary toward the small tight cluster and steal points from its edge

**Gaussian Mixture Models (GMM)** do not make this assumption — each component has its own covariance matrix. We compare both below.


In [ ]:
np.random.seed(42)

# ── Generate two clusters with very different covariances ─────────────────────
# Cluster 0: tight VIP cluster (low variance)
cov_tight  = [[0.1, 0.0], [0.0, 0.1]]           # small covariance → compact blob
X_tight    = np.random.multivariate_normal([2, 2], cov_tight, size=80)

# Cluster 1: broad mass-market cluster (high variance, elongated)
cov_spread = [[3.0, 1.5], [1.5, 2.0]]           # large covariance → spread blob
X_spread   = np.random.multivariate_normal([6, 5], cov_spread, size=200)

X_unequal  = np.vstack([X_tight, X_spread])      # combine both groups
y_true     = np.array([0]*80 + [1]*200)          # ground truth labels

# ── Fit KMeans (k=2) ──────────────────────────────────────────────────────────
km_unequal = KMeans(n_clusters=2, random_state=42, n_init=10)
y_km       = km_unequal.fit_predict(X_unequal)

# ── Fit GMM (k=2) with full covariance ───────────────────────────────────────
gmm        = GaussianMixture(n_components=2, covariance_type='full', random_state=42)
y_gmm      = gmm.fit_predict(X_unequal)

# ── Helper: align predicted labels to ground truth by majority vote ───────────
def align_labels(y_pred, y_true, n_clusters=2):
    """Flip predicted labels if needed so they best match ground truth."""
    from itertools import permutations
    best_acc, best_labels = 0, y_pred
    for perm in permutations(range(n_clusters)):
        mapping = {old: new for old, new in enumerate(perm)}
        remapped = np.array([mapping[l] for l in y_pred])
        acc = np.mean(remapped == y_true)
        if acc > best_acc:
            best_acc, best_labels = acc, remapped
    return best_labels, best_acc

y_km_aligned,  acc_km  = align_labels(y_km,  y_true)
y_gmm_aligned, acc_gmm = align_labels(y_gmm, y_true)

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette   = ['#e41a1c', '#377eb8']
titles    = ['True Clusters', f'KMeans  (acc={acc_km:.0%})', f'GMM  (acc={acc_gmm:.0%})']
label_sets = [y_true, y_km_aligned, y_gmm_aligned]

for ax, labels, title in zip(axes, label_sets, titles):
    for k in [0, 1]:
        mask = labels == k
        ax.scatter(X_unequal[mask, 0], X_unequal[mask, 1],
                   c=palette[k], s=18, alpha=0.7, label=f'Cluster {k}')
    ax.set_title(title, fontsize=12,
                 color='crimson' if 'KMeans' in title else 'darkgreen' if 'GMM' in title else 'black')
    ax.legend(fontsize=9)

fig.suptitle('Limitation 2: KMeans Struggles with Unequal Variances — GMM Handles It',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'KMeans accuracy : {acc_km:.1%}')
print(f'GMM    accuracy : {acc_gmm:.1%}')
print('\nKMeans drags its boundary toward the tight cluster, stealing points from the broad one.')


---
## Limitation 3: Sensitivity to Outliers

### The centroid is the mean — and the mean is pulled by extreme values

A centroid is computed as:

> μₖ = (1 / |Cₖ|) · Σᵢ∈Cₖ xᵢ

This is the arithmetic mean — the most outlier-sensitive measure of center. Add a single point with spend = \$50,000 to a cluster whose average spend is \$200, and the centroid jumps dramatically. That shift can:
- Pull other points across the Voronoi boundary, changing their cluster assignment
- Make the centroid no longer representative of any real customer
- Cause the cluster description to be dominated by a single anomaly

**Contrast:** the median is robust to outliers, but K-means uses the mean by design (it's required to minimise WCSS).


In [ ]:
np.random.seed(42)

# ── Clean blobs dataset (3 clusters, well separated) ─────────────────────────
X_clean, y_clean = make_blobs(n_samples=150, centers=3, cluster_std=0.8,
                               random_state=42)

# ── Add 5 extreme outlier points far from all clusters ───────────────────────
outliers = np.array([[ 8,  8],
                     [ 9,  7],
                     [-7,  8],
                     [ 8, -7],
                     [ 0, 10]])
X_dirty  = np.vstack([X_clean, outliers])   # dataset with outliers appended

# ── Fit KMeans on clean and dirty datasets ────────────────────────────────────
km_clean = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_clean)
km_dirty = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_dirty)

# ── Quantify centroid shift ───────────────────────────────────────────────────
# Sort centroids by x-coordinate so we compare the same cluster across runs
centers_clean = km_clean.cluster_centers_[np.argsort(km_clean.cluster_centers_[:, 0])]
centers_dirty = km_dirty.cluster_centers_[np.argsort(km_dirty.cluster_centers_[:, 0])]
shifts        = np.linalg.norm(centers_clean - centers_dirty, axis=1)  # Euclidean shift per centroid

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cluster_colors = ['#e41a1c', '#377eb8', '#4daf4a']

for ax, (X, km, outlier_pts, title) in zip(
        axes,
        [(X_clean, km_clean, np.zeros((0,2)), 'Clean Data'),
         (X_dirty, km_dirty, outliers,        'With 5 Outliers')]):

    # Plot regular points coloured by cluster
    n_regular = len(X_clean)                    # first 150 are always regular
    for k in range(3):
        mask = km.labels_[:n_regular] == k
        ax.scatter(X[:n_regular][mask, 0], X[:n_regular][mask, 1],
                   c=cluster_colors[k], s=25, alpha=0.7)

    # Mark outliers with large stars
    if len(outlier_pts) > 0:
        ax.scatter(outlier_pts[:, 0], outlier_pts[:, 1],
                   c='black', marker='*', s=300, zorder=6, label='Outliers')
        ax.legend(fontsize=10)

    # Mark centroids with X
    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='gold', edgecolors='black', marker='X', s=250, zorder=7, label='Centroids')
    ax.set_title(title, fontsize=12)

fig.suptitle('Limitation 3: Outliers Pull Centroids Away from True Cluster Centers',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Centroid shifts caused by 5 outliers (Euclidean distance):')
for i, s in enumerate(shifts):
    print(f'  Cluster {i}: centroid moved {s:.3f} units')
print(f'\nAverage centroid shift: {shifts.mean():.3f} units')
print('Even a handful of outliers can meaningfully distort all three centroids.')


---
## Limitation 4: Must Specify k in Advance

K-means requires you to **choose the number of clusters k before running the algorithm**. This is a chicken-and-egg problem: you need to know the structure of the data to choose k, but you need to run clustering to see the structure.

Two popular heuristics:

1. **Elbow method:** plot inertia (WCSS) vs. k. Find the "elbow" where adding more clusters gives diminishing returns. The problem: inertia **always decreases** as k increases — the elbow is subjective and sometimes absent.

2. **Silhouette score:** measures how similar a point is to its own cluster vs. the next-best cluster. Range [-1, 1]; higher is better. This is more principled but slower to compute.

We cover both methods in depth in **Notebook 07: Choosing k**. Here we just demonstrate the core problem.


In [ ]:
np.random.seed(42)

# ── Dataset with 3 true clusters ─────────────────────────────────────────────
X_k, _ = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=42)

# ── Compute inertia and silhouette for k = 2 … 10 ────────────────────────────
k_range    = range(2, 11)
inertias   = []
silhouettes = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_k)
    inertias.append(km.inertia_)                                       # sum of squared distances to centroid
    silhouettes.append(silhouette_score(X_k, labels))                 # mean silhouette coefficient

# ── Plot both metrics side by side ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Elbow plot — inertia always decreasing
ax1.plot(list(k_range), inertias, marker='o', color='steelblue', linewidth=2)
ax1.axvline(x=3, color='crimson', linestyle='--', label='True k=3')
ax1.set_xlabel('Number of clusters k', fontsize=11)
ax1.set_ylabel('Inertia (WCSS)', fontsize=11)
ax1.set_title('Elbow Plot — Inertia Always Decreases\n(elbow at k=3 is subtle)', fontsize=11)
ax1.legend()

# Silhouette plot — has a clear peak at the true k
ax2.plot(list(k_range), silhouettes, marker='s', color='darkorange', linewidth=2)
ax2.axvline(x=3, color='crimson', linestyle='--', label='True k=3')
best_k = list(k_range)[np.argmax(silhouettes)]
ax2.axvline(x=best_k, color='green', linestyle=':', label=f'Best sil. k={best_k}')
ax2.set_xlabel('Number of clusters k', fontsize=11)
ax2.set_ylabel('Silhouette Score', fontsize=11)
ax2.set_title('Silhouette Score — Peak = Best k\n(more principled)', fontsize=11)
ax2.legend()

fig.suptitle('Limitation 4: You Must Choose k — Inertia Alone Is Ambiguous',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key insight: inertia always goes down as k increases.')
print('At k = n_samples every point is its own cluster and inertia = 0.')
print('The elbow is a heuristic — silhouette score gives a more principled answer.')


---
## Limitation 5: Deterministic Local Optima

### K-means is not guaranteed to find the global optimum

K-means is a **greedy hill-climbing algorithm**:

1. Initialise k centroids (randomly or via k-means++)
2. Assign each point to its nearest centroid
3. Move each centroid to the mean of its assigned points
4. Repeat until convergence

Because step 1 is random, different initialisations lead to different local optima. The algorithm is **guaranteed to converge** but only to a **local** minimum of WCSS, not the global one.

The standard fix is to run K-means many times (`n_init` parameter) and keep the best result. **K-means++ initialisation** (the default in sklearn since v1.2) also dramatically reduces the variance across runs by choosing initial centroids that are spread apart.


In [ ]:
np.random.seed(42)

# ── Adversarial dataset: two clusters whose shape invites bad initialisation ──
# Four tight clusters arranged in a 2×2 grid — easy to mis-initialise into 2×wrong
X_adv, _ = make_blobs(n_samples=400, centers=[[-3, 0], [3, 0], [0, -3], [0, 3]],
                       cluster_std=0.6, random_state=42)

N_RUNS = 100   # number of independent K-means runs

inertias_random  = []   # random init
inertias_pp      = []   # k-means++ init

for seed in range(N_RUNS):
    # Random initialisation — prone to bad starts
    km_rand = KMeans(n_clusters=4, init='random', n_init=1, random_state=seed)
    km_rand.fit(X_adv)
    inertias_random.append(km_rand.inertia_)

    # K-means++ initialisation — spreads initial centroids intelligently
    km_pp = KMeans(n_clusters=4, init='k-means++', n_init=1, random_state=seed)
    km_pp.fit(X_adv)
    inertias_pp.append(km_pp.inertia_)

# ── Plot distribution of final inertia values ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(inertias_random, bins=30, alpha=0.6, color='tomato',    label='Random init')
ax.hist(inertias_pp,     bins=30, alpha=0.6, color='steelblue', label='K-means++ init')
ax.axvline(np.min(inertias_random), color='darkred',   linestyle='--', linewidth=1.5,
           label=f'Random best  = {np.min(inertias_random):.1f}')
ax.axvline(np.min(inertias_pp),     color='navy',      linestyle='--', linewidth=1.5,
           label=f'k-means++ best = {np.min(inertias_pp):.1f}')
ax.set_xlabel('Final Inertia (WCSS)', fontsize=11)
ax.set_ylabel('Count (out of 100 runs)', fontsize=11)
ax.set_title('Limitation 5: Local Optima — Inertia Varies Across 100 Random Starts',
             fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Random init   — mean inertia: {np.mean(inertias_random):.2f},  std: {np.std(inertias_random):.2f}')
print(f'K-means++ init — mean inertia: {np.mean(inertias_pp):.2f},  std: {np.std(inertias_pp):.2f}')
print('\nK-means++ has lower mean AND lower variance — it is almost always better than random init.')


---
## Decision Guide: When to Use K-Means vs. Alternatives

| Situation | K-Means OK? | Better Alternative |
|---|---|---|
| Clusters are compact and roughly spherical | ✅ Yes | — |
| Clusters have very different sizes/densities | ❌ No | Gaussian Mixture Model (GMM) |
| Clusters are ring-shaped or crescent-shaped | ❌ No | DBSCAN, Spectral Clustering |
| Data has many outliers | ❌ No | DBSCAN (labels outliers as noise) |
| You don't know k in advance | ⚠️ Needs elbow/silhouette | DBSCAN, HDBSCAN, Hierarchical |
| Dataset is very large (>1M rows) | ✅ Use MiniBatch variant | MiniBatchKMeans |
| You need a cluster hierarchy | ❌ No | Hierarchical Clustering |
| Clusters differ in orientation (elongated) | ❌ No | GMM with full covariance |

**Rule of thumb:** K-means is an excellent first-pass algorithm. Run it quickly to get a baseline, then ask whether the cluster shapes and sizes justify it — and if not, upgrade to a more flexible model.


In [ ]:
np.random.seed(42)

# ── Retail scenario: add VIP outlier customers ────────────────────────────────
# Simulate 200 regular customers (TotalSpend, Frequency)
regular_spend = np.random.normal(loc=300, scale=80, size=200)      # average spenders
regular_freq  = np.random.normal(loc=5,   scale=1.5, size=200)     # ~5 visits/month

# Add 5 whale customers who spend 50× the average
whale_spend   = np.array([15000, 18000, 22000, 12000, 20000], dtype=float)
whale_freq    = np.array([2, 3, 1, 4, 2], dtype=float)             # actually infrequent visitors

X_retail_reg  = np.column_stack([regular_spend, regular_freq])
X_retail_all  = np.vstack([X_retail_reg, np.column_stack([whale_spend, whale_freq])])

# ── Scale (important for KMeans to treat both features equally) ───────────────
scaler        = StandardScaler()
X_scaled_reg  = scaler.fit_transform(X_retail_reg)
X_scaled_all  = scaler.fit_transform(X_retail_all)

# ── KMeans on clean data vs data with whales ─────────────────────────────────
km_reg = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_scaled_reg)
km_all = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_scaled_all)

# Centroid shift in original feature space
centers_reg = scaler.fit_transform(X_retail_reg).mean(axis=0)      # just for reference
c_reg = km_reg.cluster_centers_                                     # (2, 2) in scaled space
c_all = km_all.cluster_centers_

# ── Plot ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (X, km, title) in zip(axes, [
        (X_scaled_reg, km_reg, 'Regular customers only'),
        (X_scaled_all, km_all, 'With 5 Whale Customers')]):

    labels = km.labels_
    for k, col in enumerate(['#377eb8', '#e41a1c']):
        mask = labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=col, s=20, alpha=0.6)

    # Mark whales in the second plot
    if 'Whale' in title:
        ax.scatter(X_scaled_all[-5:, 0], X_scaled_all[-5:, 1],
                   c='gold', marker='*', s=400, zorder=6, edgecolors='black', label='Whales')
        ax.legend()

    ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
               c='black', marker='X', s=200, zorder=7)
    ax.set_xlabel('Scaled Total Spend', fontsize=10)
    ax.set_ylabel('Scaled Frequency', fontsize=10)
    ax.set_title(title, fontsize=11)

fig.suptitle('Retail Application: Whale Customers Dominate and Distort Segments',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('With whale customers, one entire KMeans cluster is essentially just the whales.')
print('The remaining 200 regular customers are squashed into a single undifferentiated cluster.')


---
## Workarounds

When K-means limitations bite, here are practical mitigations:

### 1. Outlier removal before clustering
Use IQR clipping or isolation forest to remove extreme points before running K-means. After clustering, assign the outliers back to their nearest cluster (or keep them as an "anomaly" segment).

### 2. Log-transform skewed features
Customer spend is typically right-skewed. `np.log1p(spend)` compresses the whale values and makes the data more spherically distributed.

### 3. MiniBatchKMeans for large datasets
Standard K-means loads the full dataset into memory and requires `O(n · k · d)` operations per iteration. `MiniBatchKMeans` uses random mini-batches — much faster on large datasets with only slightly worse cluster quality.

### 4. Kernel K-means for non-convex clusters
Apply a non-linear kernel transformation (e.g., RBF kernel) that maps non-convex clusters to linearly separable clusters in a higher-dimensional space. This is computationally equivalent to Spectral Clustering.


In [ ]:
import timeit
np.random.seed(42)

# ── Generate large dataset (100k points) ─────────────────────────────────────
X_large, _ = make_blobs(n_samples=100_000, centers=8, cluster_std=1.5, random_state=42)

# ── Time KMeans vs MiniBatchKMeans ────────────────────────────────────────────
def run_kmeans():
    KMeans(n_clusters=8, random_state=42, n_init=3, max_iter=100).fit(X_large)

def run_minibatch():
    MiniBatchKMeans(n_clusters=8, random_state=42, n_init=3,
                    batch_size=1024, max_iter=100).fit(X_large)    # processes 1024 points per step

# Run 3 repeats of each, take the minimum (most stable estimate)
t_km  = min(timeit.repeat(run_kmeans,    number=1, repeat=3))
t_mb  = min(timeit.repeat(run_minibatch, number=1, repeat=3))

# ── Also compare final inertia to check quality trade-off ────────────────────
km_full = KMeans(n_clusters=8, random_state=42, n_init=3, max_iter=100).fit(X_large)
km_mini = MiniBatchKMeans(n_clusters=8, random_state=42, n_init=3,
                           batch_size=1024, max_iter=100).fit(X_large)

# ── Bar chart: speed comparison ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(['KMeans', 'MiniBatchKMeans'], [t_km, t_mb],
        color=['steelblue', 'darkorange'], width=0.4)
ax1.set_ylabel('Time (seconds)', fontsize=11)
ax1.set_title(f'Speed on 100k Points\nSpeedup: {t_km/t_mb:.1f}×', fontsize=11)
for bar, val in zip(ax1.patches, [t_km, t_mb]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.2f}s', ha='center', fontsize=10)

ax2.bar(['KMeans', 'MiniBatchKMeans'], [km_full.inertia_, km_mini.inertia_],
        color=['steelblue', 'darkorange'], width=0.4)
ax2.set_ylabel('Final Inertia (WCSS)', fontsize=11)
ax2.set_title('Quality Trade-off\n(higher inertia = slightly worse clusters)', fontsize=11)
quality_loss = (km_mini.inertia_ - km_full.inertia_) / km_full.inertia_ * 100
ax2.set_title(f'Quality Trade-off\nMiniBatch inertia is {quality_loss:.1f}% higher', fontsize=11)

fig.suptitle('Workaround: MiniBatchKMeans — Faster with Minimal Quality Loss',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'KMeans:          {t_km:.2f}s,  inertia = {km_full.inertia_:,.0f}')
print(f'MiniBatchKMeans: {t_mb:.2f}s,  inertia = {km_mini.inertia_:,.0f}')
print(f'Speedup: {t_km/t_mb:.1f}× faster  |  Quality loss: {quality_loss:.1f}%')


---
## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** You have customer data that you believe forms two segments: a small group of weekly high-spenders (tight cluster) and a large group of monthly casual shoppers (loose, spread-out cluster). You run K-means with k=2 and the results look wrong. What limitation is most likely causing this, and what algorithm would you try instead?

<details><summary>Show answer</summary>

This is **Limitation 2: equal-variance assumption**. K-means assumes all clusters have roughly the same spread (covariance). Here one cluster is tight and one is broad, so K-means will draw its boundary incorrectly — typically placing it too close to the tight cluster and stealing its edge points.

**Alternative:** Gaussian Mixture Model (GMM) with `covariance_type='full'`. GMM allows each cluster to have its own covariance matrix, so it can represent both the tight VIP cluster and the broad casual cluster accurately.

</details>

---

**Question 2:** You run K-means three times on the same dataset with `n_init=1` and get three different inertia values: 1240, 890, and 1050. Which result should you use, and how does sklearn's default `n_init=10` help?

<details><summary>Show answer</summary>

You should use the result with **inertia = 890** (the lowest), because K-means minimises WCSS (inertia) — the lower, the better the solution.

The different values show that K-means converged to different **local optima** depending on the random initialisation. This is Limitation 5.

With `n_init=10` (sklearn default), K-means runs 10 times with different random initialisations and automatically keeps the run with the lowest inertia. This makes the result much more stable and likely to find the global optimum.

</details>

---

**Question 3:** Your colleague says "I plotted the elbow curve and the inertia drops sharply from k=1 to k=5, then slows down — so the answer is clearly k=5." What is the weakness of this argument, and what additional metric would you compute to validate the choice?

<details><summary>Show answer</summary>

**Weakness:** Inertia **always** decreases as k increases — at k = n_samples every point is its own cluster with inertia = 0. The "elbow" is visual and subjective; in real data the curve is often smooth with no clear elbow.

**Additional metric:** Compute the **silhouette score** for each k. The silhouette score measures how well-separated clusters are (range [-1, 1]; higher = better). Unlike inertia, silhouette score has a natural maximum — the k that maximises silhouette score is a principled choice. A Gap Statistic or BIC (for GMM) are also valid alternatives.

</details>


---
## Key Takeaways

| Limitation | Root cause | Practical fix |
|---|---|---|
| Non-convex clusters | Linear Voronoi boundaries | DBSCAN, Spectral Clustering |
| Unequal variance | Spherical covariance assumption | Gaussian Mixture Model |
| Outlier sensitivity | Centroid = arithmetic mean | Remove outliers first; use DBSCAN |
| Must specify k | No built-in model selection | Elbow + silhouette; DBSCAN |
| Local optima | Greedy hill-climbing | `n_init > 1`; k-means++ init |

**The bottom line:** K-means is fast, interpretable, and often a perfectly good first-pass algorithm. Use it when:
- You expect roughly round, similarly-sized clusters
- You have cleaned your data (no extreme outliers)
- You have a method to choose k (silhouette, domain knowledge)
- Speed matters (especially with MiniBatchKMeans)

When those conditions do not hold, upgrade to a more flexible algorithm. The next notebooks cover hierarchical clustering (NB04), DBSCAN (NB05), and GMM (NB06).

---
*Next: Notebook 04 — Hierarchical Clustering*
